In [ ]:
from pyspark.sql import SparkSession

# Creating Spark Session
spark = (SparkSession.builder
    .appName("BigData_Lab02.1_Ubuntu") # Spark session name
    .master("local[*]") \
    .config("spark.driver.bindAddress", "127.0.0.1")\
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.memory", "2g")
    .config("spark.jars.packages", 
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1,"
            "org.apache.kafka:kafka-clients:3.6.0,"
            "org.apache.spark:spark-streaming-kafka-0-10_2.13:4.0.1") 
    .getOrCreate())

# Checking success
print(f"Successfully! Spark version: {spark.version}")

In [ ]:
spark.stop()

In [ ]:
import kagglehub

# Download dataset from kaggle
path = kagglehub.dataset_download("grouplens/movielens-latest-small")

# reading data to spark frame
df_ratings = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_movies = spark.read.csv(path + "/movies.csv", header=True, inferSchema=True)
df_tags = spark.read.csv(path + "/tags.csv", header=True, inferSchema=True)

# Checking by show 1 row of each df
df_ratings.show(1)
df_movies.show(1)
df_tags.show(1)

In [ ]:
from confluent_kafka.admin import AdminClient

admin_client = AdminClient({
    'bootstrap.servers': 'localhost:9092,localhost:9192,localhost:9292'
})

topics_to_delete = ["Lab1_ratings", "Lab1_movies", "Lab1_tags"]

fs = admin_client.delete_topics(topics_to_delete)

for topic, f in fs.items():
    try:
        f.result()
        print(f"Topic '{topic}' deleted.")
    except Exception as e:
        print(f"Error deleting topic {topic}: {e}")

In [ ]:
from confluent_kafka.admin import AdminClient, NewTopic

# Connect to brokers
admin_client = AdminClient({'bootstrap.servers': 'localhost:9092,localhost:9192,localhost:9292'})

# Setting
new_topics = [
    NewTopic(topic="Lab1_ratings", num_partitions=1, replication_factor=3),
    NewTopic(topic="Lab1_movies", num_partitions=1, replication_factor=3),
    NewTopic(topic="Lab1_tags", num_partitions=1, replication_factor=3)
]

# Creating topics
fs = admin_client.create_topics(new_topics)

for topic, f in fs.items():
    try:
        f.result()
        print(f"Topic '{topic}' created.")
    except Exception as e:
        print(f"Error when creating topic {topic}: {e}")

In [ ]:
from pyspark.sql.functions import struct, to_json
import time
# Kafka bootstrap servers
brokers = "localhost:9092,localhost:9192,localhost:9292"

def push_to_kafka_streaming(df, topic_name, batch_size=100, delay=2):
    # Chuyển DataFrame thành danh sách các chuỗi JSON
    json_list = df.selectExpr("to_json(struct(*)) AS value").collect()
    
    print(f">>> Bắt đầu đẩy dữ liệu vào topic: {topic_name}")
    for i in range(0, len(json_list), batch_size):
        # Lấy một nhóm (batch) dữ liệu
        batch = json_list[i:i+batch_size]
        batch_df = spark.createDataFrame(batch)
        
        # Ghi nhóm này vào Kafka
        batch_df.write \
            .format("kafka") \
            .option("kafka.bootstrap.servers", brokers) \
            .option("topic", topic_name) \
            .save()
        
        print(f"Đã đẩy {i + len(batch)} dòng...")
        time.sleep(delay) # Nghỉ 2 giây để giả lập dữ liệu đang đổ về chậm

push_to_kafka_streaming(df_ratings, "Lab1_ratings")
push_to_kafka_streaming(df_movies, "Lab1_movies")
push_to_kafka_streaming(df_tags, "Lab1_tags")

print("Data push to kafka successfully!")

## EX2

## EX3

In [ ]:
from pyspark.sql.window import Window

# 1. Thực hiện aggregation trên stream (Tính tổng count theo window)
# Sử dụng 5-minute tumbling window và watermark 10 phút [cite: 209, 212]
windowed_counts = ratings_stream \
    .withWatermark("timestamp", "10 minutes") \
    .groupBy(window(col("timestamp"), "5 minutes"), col("movieId")) \
    .count()

# 2. Join với dữ liệu movies tĩnh để lấy title [cite: 208, 209]
trending_df = windowed_counts.join(movies_static, "movieId")

# 3. Định nghĩa hàm xử lý cho mỗi batch (Đây là nơi xử lý Rank)
def process_top_k(batch_df, batch_id):
    # Trong đây, batch_df là một STATIC DataFrame
    window_spec = Window.partitionBy("window").orderBy(col("count").desc(), col("movieId").asc())
    
    # Thực hiện xếp hạng và lọc Top 3 [cite: 209, 210, 213]
    final_batch = batch_df.withColumn("rank", dense_rank().over(window_spec)) \
        .filter(col("rank") <= 3) \
        .select("window", "movieId", "title", "count", "rank") \
        .orderBy("window", "rank")
    
    print(f"Batch ID: {batch_id} - Trending Now (Top 3 per window)")
    final_batch.show(truncate=False)

# 4. Khởi chạy stream với foreachBatch
query3 = trending_df.writeStream \
    .foreachBatch(process_top_k) \
    .trigger(processingTime='5 seconds') \
    .start()

query3.awaitTermination(20)
query3.stop()

## EX4

In [ ]:
from pyspark.sql.functions import *

# 1. Thiết lập Watermark cho cả hai luồng (Bắt buộc cho Stream-Stream Join) [cite: 220, 224]
# Cần đặt watermark để Spark biết khi nào có thể xóa các trạng thái cũ trong bộ nhớ [cite: 34, 222]
ratings_with_wm = ratings_stream.withWatermark("timestamp", "10 minutes")
tags_with_wm = tags_stream.withWatermark("timestamp", "10 minutes")

# 2. Thực hiện Stream-Stream Join trên movieId [cite: 216, 225]
# Đối với Stream-Stream join, bạn PHẢI có một điều kiện ràng buộc về thời gian 
# Ở đây chúng ta tìm các tag được gán trong khoảng +/- 5 phút so với thời điểm rating [cite: 215]
stream_stream_join = ratings_with_wm.alias("r").join(
    tags_with_wm.alias("t"),
    expr("""
        r.movieId = t.movieId AND
        t.timestamp >= r.timestamp - interval 5 minutes AND
        t.timestamp <= r.timestamp + interval 5 minutes
    """),
    "inner"
)

# 3. Chọn và định dạng các cột đầu ra theo yêu cầu [cite: 221]
# Cấu trúc: (movieId, tag, rating, ratingTime, tagTime)
result_ex4 = stream_stream_join.select(
    col("r.movieId"),
    col("t.tag"),
    col("r.rating"),
    col("r.timestamp").alias("ratingTime"),
    col("t.timestamp").alias("tagTime")
)

# 4. Ghi kết quả ra console mỗi 5 giây ở chế độ APPEND [cite: 216, 217, 221]
# Lưu ý: Stream-stream join không hỗ trợ Complete mode, chỉ dùng Append hoặc Update 
query4 = result_ex4.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", "false") \
    .trigger(processingTime='5 seconds') \
    .start()

# 5. GIỚI HẠN THỜI GIAN CHẠY 20 GIÂY
print(">>> Đang thực hiện Stream-Stream Join (Ratings & Tags) trong 20 giây...")
query4.awaitTermination(20)

# Dừng query sau khi hết 20 giây demo
query4.stop()
print(">>> Demo Exercise 4 đã dừng thành công.")